In [1]:
import pandas as pd

file = "B3LYP.csv"

df = pd.read_csv(file, encoding="utf-8-sig")

df["ΔEST-PM7"] = pd.to_numeric(df["ΔEST-PM7"], errors="coerce")
df["ΔEST_ad_eV"] = pd.to_numeric(df["ΔEST_ad_eV"], errors="coerce")

valid = df[["ΔEST-PM7", "ΔEST_ad_eV"]].dropna()

rho = valid["ΔEST-PM7"].corr(
    valid["ΔEST_ad_eV"],
    method="spearman"
)

print(rho)

0.5583307425148151


In [2]:
import pandas as pd

file = "MN15.csv"

df = pd.read_csv(file, encoding="utf-8-sig")

df["ΔEST-PM7"] = pd.to_numeric(df["ΔEST-PM7"], errors="coerce")
df["ΔEST_ad_eV"] = pd.to_numeric(df["ΔEST_ad_eV"], errors="coerce")

valid = df[["ΔEST-PM7", "ΔEST_ad_eV"]].dropna()

rho = valid["ΔEST-PM7"].corr(
    valid["ΔEST_ad_eV"],
    method="spearman"
)

print(rho)

0.5239169650896326


In [3]:
import pandas as pd

file = "B3LYP.csv"

df = pd.read_csv(file, encoding="utf-8-sig")

df["ΔEST-PM7"] = pd.to_numeric(df["ΔEST-PM7"], errors="coerce")
df["ΔEST_ad_eV"] = pd.to_numeric(df["ΔEST_ad_eV"], errors="coerce")

valid = df[["ΔEST-PM7", "ΔEST_ad_eV"]].dropna()

MAD = (valid["ΔEST_ad_eV"] - valid["ΔEST-PM7"]).abs().mean()

print(MAD)

0.4281156352271197


In [4]:
import pandas as pd

file = "MN15.csv"

df = pd.read_csv(file, encoding="utf-8-sig")

df["ΔEST-PM7"] = pd.to_numeric(df["ΔEST-PM7"], errors="coerce")
df["ΔEST_ad_eV"] = pd.to_numeric(df["ΔEST_ad_eV"], errors="coerce")

valid = df[["ΔEST-PM7", "ΔEST_ad_eV"]].dropna()

MAD = (valid["ΔEST_ad_eV"] - valid["ΔEST-PM7"]).abs().mean()

print(MAD)

0.21274215646140918


In [5]:
import pandas as pd
from scipy.stats import spearmanr


b3lyp_file = "B3LYP.csv"


mn15_file = "MN15.csv"


b3lyp = pd.read_csv(b3lyp_file, encoding="utf-8-sig")
mn15 = pd.read_csv(mn15_file, encoding="utf-8-sig")


b3lyp["ΔEST_ad_eV"] = pd.to_numeric(b3lyp["ΔEST_ad_eV"], errors="coerce")
mn15["ΔEST_ad_eV"] = pd.to_numeric(mn15["ΔEST_ad_eV"], errors="coerce")


merged = pd.merge(
    b3lyp[["est", "ΔEST_ad_eV"]],
    mn15[["est", "ΔEST_ad_eV"]],
    on="est",
    how="inner",
    suffixes=("_B3LYP", "_MN15")
)


valid = merged.dropna(
    subset=["ΔEST_ad_eV_B3LYP", "ΔEST_ad_eV_MN15"]
).copy()

# N
N = len(valid)

# Spearman rho
rho, p_value = spearmanr(
    valid["ΔEST_ad_eV_B3LYP"],
    valid["ΔEST_ad_eV_MN15"]
)

# MAD = mean absolute difference
MAD = (
    valid["ΔEST_ad_eV_B3LYP"]
    - valid["ΔEST_ad_eV_MN15"]
).abs().mean()

# MSD = B3LYP - MN15
MSD = (
    valid["ΔEST_ad_eV_B3LYP"]
    - valid["ΔEST_ad_eV_MN15"]
).mean()

print(f"N = {N}")
print(f"Spearman rho = {rho:.4f}")
print(f"MAD (eV) = {MAD:.4f}")
print(f"MSD (eV) = {MSD:.4f}")

N = 165
Spearman rho = 0.8486
MAD (eV) = 0.4566
MSD (eV) = -0.4542


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr


b3lyp_file = "B3LYP.csv"


mn15_file = "MN15.csv"


b3lyp = pd.read_csv(b3lyp_file, encoding="utf-8-sig")
mn15 = pd.read_csv(mn15_file, encoding="utf-8-sig")


properties = {
    "k_r": "k_r_s-1",
    "k_nr": "k_nr_s-1",
    "k_ISC": "k_isc_s-1",
    "k_RISC": "k_risc_s-1",
}

results = []

for label, col in properties.items():

    
    b3 = b3lyp[["est", col]].copy()
    mn = mn15[["est", col]].copy()

    b3[col] = pd.to_numeric(b3[col], errors="coerce")
    mn[col] = pd.to_numeric(mn[col], errors="coerce")

    
    merged = pd.merge(
        b3,
        mn,
        on="est",
        how="inner",
        suffixes=("_B3LYP", "_MN15")
    )

    col_b3 = f"{col}_B3LYP"
    col_mn = f"{col}_MN15"


    valid = merged[
        np.isfinite(merged[col_b3]) &
        np.isfinite(merged[col_mn]) &
        (merged[col_b3] > 0) &
        (merged[col_mn] > 0)
    ].copy()

    # N
    N = len(valid)

    # Spearman rho
    rho, p_value = spearmanr(
        valid[col_b3],
        valid[col_mn]
    )

    # |Δlog10 k| = |log10(k_B3LYP) - log10(k_MN15)|
    delta_log10 = np.abs(
        np.log10(valid[col_b3]) -
        np.log10(valid[col_mn])
    )

    # Median |Δlog10 k|
    median_delta_log10 = np.median(delta_log10)

    results.append({
        "Property": label,
        "N": N,
        "Spearman rho": rho,
        "Median |Δlog10 k|": median_delta_log10
    })



result_df = pd.DataFrame(results)

print(result_df.to_string(
    index=False,
    formatters={
        "Spearman rho": "{:.3f}".format,
        "Median |Δlog10 k|": "{:.2f}".format
    }
))

Property   N Spearman rho Median |Δlog10 k|
     k_r 146        0.630              1.35
    k_nr 166        0.460              0.62
   k_ISC 165        0.043              3.01
  k_RISC 141       -0.105              4.81


In [7]:
import pandas as pd
import numpy as np

# ============================================================
# File paths
# ============================================================


b3lyp_file = "B3LYP.csv"


mn15_file = "MN15.csv"


# ============================================================
# Read data
# ============================================================

b3lyp = pd.read_csv(b3lyp_file, encoding="utf-8-sig")
mn15 = pd.read_csv(mn15_file, encoding="utf-8-sig")


# ============================================================
# Columns used in Table S13
# ============================================================

properties = {
    "SOC (cm^-1)": "SOC_cm-1",
    "k_r (s^-1)": "k_r_s-1",
    "k_ISC (s^-1)": "k_isc_s-1",
    "k_RISC (s^-1)": "k_risc_s-1",
}


# ============================================================
# Analysis function
# ============================================================

def analyze_property(df, column):

    
    values = pd.to_numeric(df[column], errors="coerce")

    
    finite = values[np.isfinite(values)]

    N_valid = len(finite)

    
    positive = finite[finite > 0]

    if len(positive) > 0:
        median = positive.median()
        q1 = positive.quantile(0.25)
        q3 = positive.quantile(0.75)
    else:
        median = np.nan
        q1 = np.nan
        q3 = np.nan

    
    zero_n = (finite == 0).sum()
    negative_n = (finite < 0).sum()

    
    zero_pct = zero_n / N_valid * 100 if N_valid > 0 else np.nan
    negative_pct = negative_n / N_valid * 100 if N_valid > 0 else np.nan

    
    unavailable_n = len(values) - N_valid

    return {
        "N_valid": N_valid,
        "Median": median,
        "Q1": q1,
        "Q3": q3,
        "Zero_N": zero_n,
        "Zero_%": zero_pct,
        "Negative_N": negative_n,
        "Negative_%": negative_pct,
        "Unavailable": unavailable_n
    }


# ============================================================
# Calculate
# ============================================================

results = []

for property_name, column in properties.items():

    for method_name, df in [
        ("B3LYP", b3lyp),
        ("MN15", mn15)
    ]:

        r = analyze_property(df, column)

        results.append({
            "Property": property_name,
            "Method": method_name,
            **r
        })


result_df = pd.DataFrame(results)


# ============================================================
# Print results
# ============================================================

for _, row in result_df.iterrows():

    print("=" * 70)

    print(f"Property: {row['Property']}")
    print(f"Method:   {row['Method']}")

    print(f"N_valid = {int(row['N_valid'])}")

    print(f"Median = {row['Median']:.6e}")

    print(
        f"IQR = {row['Q1']:.6e} - "
        f"{row['Q3']:.6e}"
    )

    print(
        f"Zero N (%) = "
        f"{int(row['Zero_N'])} "
        f"({row['Zero_%']:.1f}%)"
    )

    print(
        f"Negative N (%) = "
        f"{int(row['Negative_N'])} "
        f"({row['Negative_%']:.1f}%)"
    )

    print(
        f"Unavailable = "
        f"{int(row['Unavailable'])}"
    )

Property: SOC (cm^-1)
Method:   B3LYP
N_valid = 193
Median = 3.316625e-02
IQR = 1.732051e-02 - 7.952562e-02
Zero N (%) = 35 (18.1%)
Negative N (%) = 0 (0.0%)
Unavailable = 0
Property: SOC (cm^-1)
Method:   MN15
N_valid = 183
Median = 1.407125e-01
IQR = 5.290659e-02 - 3.959774e-01
Zero N (%) = 8 (4.4%)
Negative N (%) = 0 (0.0%)
Unavailable = 2
Property: k_r (s^-1)
Method:   B3LYP
N_valid = 190
Median = 1.982442e+03
IQR = 7.440628e+01 - 3.348364e+04
Zero N (%) = 17 (8.9%)
Negative N (%) = 0 (0.0%)
Unavailable = 3
Property: k_r (s^-1)
Method:   MN15
N_valid = 185
Median = 3.985380e+04
IQR = 1.150334e+03 - 1.005723e+06
Zero N (%) = 13 (7.0%)
Negative N (%) = 0 (0.0%)
Unavailable = 0
Property: k_ISC (s^-1)
Method:   B3LYP
N_valid = 193
Median = 6.763993e+03
IQR = 4.280038e+01 - 4.580828e+05
Zero N (%) = 0 (0.0%)
Negative N (%) = 0 (0.0%)
Unavailable = 0
Property: k_ISC (s^-1)
Method:   MN15
N_valid = 183
Median = 2.645291e+04
IQR = 1.841424e+02 - 2.855372e+06
Zero N (%) = 0 (0.0%)
Negative 